# Experiments 84
Combination of hyperparameters that provided the best results.

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º ***(v5i)***
    - Plus soil images
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Experiments:**
    1. Mix of hyperparms:
        - 9x augmentation *(synthetic data)*
        - Full Fine-Tuning *(No freeze)*
        - Max epochs (4 hours)
        -
        - IOU/CONF optimization _(on valid)_

## Init

In [1]:
import os
import shutil
import fnmatch
import pickle

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

## Helper Functions

In [3]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [4]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [5]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [6]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [7]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [8]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

### Validation functions

In [9]:
cm = lambda results: results.confusion_matrix.matrix.tolist() if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix') else None

In [10]:
import json

def save_json(results):
  try:
    data_to_store = {
        "confusion_matrix": cm(results),
        "results_dict": results.results_dict,
        "speed": results.speed
    }

    # Convert the Python dictionary to a JSON string
    json_data = json.dumps(data_to_store, indent=4)

    # You can now save this JSON string to a file
    folder = str(results.save_dir)
    with open(f"/content/{folder}/results.json", "w") as f:
        f.write(json_data)

    print("✅ JSON file stored in:", folder)

  except Exception as e:
      print(f"❌ An error occurred: {e}")

  #return json_data


In [11]:
def gimme_metrics(results):
  matrix = cm(results)
  total_det = sum(sum(value) for value in matrix)
  percentages = []
  for row in matrix:
      values_percentages = []
      for value in row:
          if total_det != 0:
              percentage = (value / total_det) * 100
          else:
              percentage = 0.0
          values_percentages.append(f"{percentage:.2f}%")
      percentages.append(values_percentages)

  print("Total objects detected:", total_det)
  print("Confusion matrix:")
  for row in percentages:
      print(row)

  return matrix


In [12]:
def show_cm(TP, FP, FN):
    matrix = [[TP, FP], [FN, 0]]
    total_det = sum(sum(value) for value in matrix)
    percentages = []
    for row in matrix:
        values_percentages = []
        for value in row:
            if total_det != 0:
                percentage = (value / total_det) * 100
            else:
                percentage = 0.0
            values_percentages.append(f"{percentage:.2f}%")
        percentages.append(values_percentages)

    print("Total objects detected:", total_det)
    print("\nConfusion matrix:")
    for row in percentages:
        a, b = row
        print(f"[ {a} , {b} ]")

In [13]:
def show_metrics(TP, FP, FN):
    show_cm(TP, FP, FN)
    accuracy = TP/(TP+FP+FN)
    precision = TP/(TP+FP)
    recall = TP/(TP+FN)
    f1 = 2 * (precision * recall) / (precision + recall)
    f2 = 1.25 * (precision * recall) / (0.25 * precision + recall)
    fm = (precision * recall) ** 0.5
    print("\nMetrics:")
    print(f"- Accuracy: {accuracy:.3f}")
    print(f"- Precision: {precision:.3f}")
    print(f"- Recall: {recall:.3f}")
    print(f"- F1 Score: {f1:.3f}")
    print(f"- F½ Score: {f2:.3f}")
    print(f"- G-mean: {fm:.3f}")

# Datasets builder

## Importing from Drive

In [21]:
!rm -rf /content/sample_data

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

3.5m.v3i.yolov8.640px		       3.5m.v4i.yolov8_blended.640px
3.5m.v3i.yolov8.640px.aug.v1	       3.5m.v4i.yolov8_blended.640px.aug.v1
3.5m.v3i.yolov8.640px.aug.v1.soil_aug  3.5m.v5i.yolov8.640px-2steps.aug2
3.5m.v3i.yolov8.640px_clahe	       best_e26.pt
3.5m.v3i.yolov8.640px.soil_aug	       best_e50.pt
3.5m.v4i.yolov8.640px		       Inference
3.5m.v4i.yolov8.640px_209	       models
3.5m.v4i.yolov8.640px.aug.v1	       optuna_yolov8_f1_study.db


In [23]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 16 dataset options:


['3.5m.v3i.yolov8.640px',
 'Inference',
 'models',
 '3.5m.v3i.yolov8.640px.aug.v1',
 'best_e26.pt',
 'optuna_yolov8_f1_study.db',
 '3.5m.v3i.yolov8.640px.soil_aug',
 '3.5m.v3i.yolov8.640px.aug.v1.soil_aug',
 '3.5m.v3i.yolov8.640px_clahe',
 '3.5m.v4i.yolov8.640px',
 '3.5m.v4i.yolov8_blended.640px',
 '3.5m.v4i.yolov8.640px_209',
 '3.5m.v4i.yolov8_blended.640px.aug.v1',
 'best_e50.pt',
 '3.5m.v4i.yolov8.640px.aug.v1',
 '3.5m.v5i.yolov8.640px-2steps.aug2']

**For this experiments:** `3.5m.v5i.yolov8.640px-2steps.aug2`

In [24]:
choose_dataset = 16
index = choose_dataset - 1
model_name = os.listdir(drive_path)[index]
print("Chosen model:", model_name)

Chosen model: 3.5m.v5i.yolov8.640px-2steps.aug2


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [25]:
# Option 2 (download just the needed dataset)
cloud_path = f"/content/drive/MyDrive/YOLO/{model_name}/"
local_path = f"/content/YOLO/"
!mkdir $local_path
!cp -r $cloud_path $local_path

In [26]:
src_folder = f"/content/YOLO/{model_name}"
data = f"{src_folder}/data.yaml"
data

'/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/data.yaml'

## Download model

In [15]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [16]:
# Load pretrain YOLO v8 model
model = YOLO("yolov8m.pt")

100%|██████████| 49.7M/49.7M [00:00<00:00, 128MB/s]


# Finetuning

In [27]:
print("FINISHED!!")

FINISHED!!


### Optimization

In [17]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [18]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [19]:
!nvidia-smi

Wed May 14 22:21:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
!yolo version

8.3.134


-----
## Experiment FINAL
### *YOLOv8 Mid | Mix of hyperparms*
Carefully disabling Ultralytics default augmentation.
1. Generated by Roboflow (2-steps)

Process applied:
- Saturation: ±30%
- Brightness: ±25%
- Exposure: ±5%
- Rotation: clockwise/counter/upside-down
- Flip: H/V
- Crop (zoom): 0-30%
- Blur: 2px
- Noise: 0.1%

### Train

In [28]:
# Garbage collection
import gc
for i in range(10):
  torch.cuda.empty_cache()
  gc.collect()

In [29]:
# Set's maximum training time (in hours)
time: float = 4 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [30]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=1000,
    val=True,
    imgsz=640,
    batch=-1,
    patience=500,
    time = time,
    multi_scale=True,
    weight_decay=0.0015,
    dropout=0.05,
    momentum=0.99,
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.05, dynamic=False, embed=None, epochs=1000, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=500, perspective=0.0, plo

100%|██████████| 755k/755k [00:00<00:00, 21.2MB/s]

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

 16                  -1  1    332160  ultralytics.nn.modules.conv.Conv             [192, 192, 3, 2]              
 17            [-1, 12]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 18                  -1  2   1846272  ultralytics.nn.modules.block.C2f             [576, 384, 2]                 
 19                  -1  1   1327872  ultralytics.nn.modules.conv.Conv             [384, 384, 3, 2]              
 20             [-1, 9]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  2   4207104  ultralytics.nn.modules.block.C2f             [960, 576, 2]                 
 22        [15, 18, 21]  1   3776275  ultralytics.nn.modules.head.Detect           [1, [192, 384, 576]]          
Model summary: 169 layers, 25,856,899 parameters, 25,856,883 gradients, 79.1 GFLOPs

Transferred 469/475 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mi

100%|██████████| 5.35M/5.35M [00:00<00:00, 94.8MB/s]


...

    124/124      12.6G      1.447     0.7505      1.113        271        704:  22%|██▏       | 30/136 [00:17<01:00,  1.75it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  1.83it/s]

                   all        108       3467       0.58      0.552      0.528       0.18



124 epochs completed in 4.001 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train/weights/best.pt, 52.0MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:05<00:00,  1.70s/it]


                   all        108       3467      0.607      0.529      0.545      0.195
Speed: 0.2ms preprocess, 11.5ms inference, 0.0ms loss, 5.0ms postprocess per image
Results saved to runs/detect/train


In [31]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c34ad03d6d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [32]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=1000,
          time=4,
          patience=500,
          batch=19,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.05,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          

In [33]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train


### Save results

In [34]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save1/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save1/


### Validation

In [35]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [36]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1863.9±715.0 MB/s, size: 89.3 KB)


val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:07<00:00,  3.79s/it]


                   all        108       3467       0.59      0.564      0.572      0.225
Speed: 9.1ms preprocess, 22.6ms inference, 0.0ms loss, 2.8ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val


In [37]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val


In [38]:
save_json(results)

✅ JSON file stored in: runs/detect/val


In [39]:
matrix = gimme_metrics(results)

Total objects detected: 4768.0
Confusion matrix:
['45.93%', '27.29%']
['26.78%', '0.00%']


### Save results

In [40]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save1/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save1/


### Metrics

In [41]:
matrix

[[2190.0, 1301.0], [1277.0, 0.0]]

In [42]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4768.0

Confusion matrix:
[ 45.93% , 27.29% ]
[ 26.78% , 0.00% ]

Metrics:
- Accuracy: 0.459
- Precision: 0.627
- Recall: 0.632
- F1 Score: 0.629
- F½ Score: 0.628
- G-mean: 0.629


In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4768.0

Confusion matrix:
[ 45.93% , 27.29% ]
[ 26.78% , 0.00% ]

Metrics:
- Accuracy: 0.459
- Precision: 0.627
- Recall: 0.632
- F1 Score: 0.629
- F½ Score: 0.628
- G-mean: 0.629


In [ ]:
['44.07%', '28.41%']
['27.52%', '0.00%']